In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
# Select an available font that can display Chinese labels.
from matplotlib import font_manager
_available_fonts = {f.name for f in font_manager.fontManager.ttflist}
for _font in ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Arial Unicode MS"]:
    if _font in _available_fonts:
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
from pathlib import Path
file_path = Path("../RawData for PC1-PC2-Cap.xlsx")


In [ ]:
df = pd.read_excel(file_path)
print(f"Dataset dimensions：{df.shape[0]} rows × {df.shape[1]} columns")
print("=" * 60)
display(df.head(15))

In [ ]:
# ====================== 2. Extract the independent variables and response values. ======================
x = df["PC1"].values  # PC1 as x
y = df["PC2"].values  # PC2 as y
z_true = df["Cap."].values  # Experimental response values z

print(f"\nPC1(x) range：{x.min():.4f} ~ {x.max():.4f}")
print(f"PC2(y) range：{y.min():.4f} ~ {y.max():.4f}")
print(f"Cap.(z) range：{z_true.min():.4f} ~ {z_true.max():.4f}")

In [ ]:
 #====================== 3. Construct all terms for the quadratic response surface ======================#
# Define basic variables
df["x"] = df["PC1"]
df["y"] = df["PC2"]
df["z"] = df["Cap."]

# Construct quadratic and interaction terms
df["x2"] = df["x"] ** 2
df["y2"] = df["y"] ** 2
df["xy"] = df["x"] * df["y"]

In [ ]:
# ============ 4. Least-squares fitting of the quadratic response surface model ======================
# Model equation：z = b0 + b1*x + b2*y + b3*x2 + b4*y2 + b5*xy
model_formula = "z ~ x + y + x2 + y2 + xy"
ols_model = ols(model_formula, data=df).fit()

# Calculate predicted values and residuals
z_true = df["z"].values
z_pred = ols_model.fittedvalues.values
residual = z_true - z_pred

In [ ]:
# ====================== 5. Overall model fitting evaluation ======================
print("\n" + "="*70)
print("                Quadratic Response Surface Model Fitting Evaluation")
print("="*70)

# Output the fitted quadratic response surface equation
coef = ols_model.params
print("\n【Optimized Quadratic Response Surface Equation】")
print(f"z = {coef['Intercept']:.4f} + {coef['x']:.4f}x + {coef['y']:.4f}y + {coef['x2']:.4f}x² + {coef['y2']:.4f}y² + {coef['xy']:.4f}xy")

In [ ]:
# Calculate basic statistical parameters
n = len(z_true)
term_num = 5
r2 = r2_score(z_true, z_pred)
adj_r2 = ols_model.rsquared_adj
mse = mean_squared_error(z_true, z_pred)
rmse = np.sqrt(mse)
df_error = ols_model.df_resid

print("\n【Overall Goodness-of-Fit Statistics】")
print(f"Coefficient of determination R²          = {r2:.6f}")
print(f"Adjusted coefficient of determination Adj-R²   = {adj_r2:.6f}")
print(f"Mean squared error MSE          = {mse:.6f}")
print(f"Root mean squared error RMSE       = {rmse:.6f}")
print(f"Total number of observations n            = {n}")
print(f"Residual degrees of freedom df_error   = {df_error:.0f}")

In [ ]:
# 5.1 ANOVA table
ss_total = np.sum((z_true - np.mean(z_true)) ** 2)
ss_model = np.sum((z_pred - np.mean(z_true)) ** 2)
ss_error = np.sum(residual ** 2)

df_model = term_num
df_error = n - term_num - 1
df_total = n - 1

ms_model = ss_model / df_model
ms_error = ss_error / df_error
F = ms_model / ms_error
p_F = 1 - stats.f.cdf(F, df_model, df_error)

print("\n【ANOVA Table】")
anova_table = pd.DataFrame({
    "Sum of Squares SS": [ss_model, ss_error, ss_total],
    "Degrees of Freedom df": [df_model, df_error, df_total],
    "Mean Square MS": [ms_model, ms_error, np.nan],
    "F-value": [F, np.nan, np.nan],
    "P-value": [p_F, np.nan, np.nan]
}, index=["Regression Model", "Residual Error", "Total"])
display(anova_table)

In [ ]:
# ====================== 6. Overall ANOVA table ======================
print("\n【Overall ANOVA Results】")
anova_total = sm.stats.anova_lm(ols_model, typ=1)
display(anova_total)


In [ ]:
# ====================== 7. Individual-term ANOVA (Type III Partial Sum of Squares) ======================
# Type III SS: Partial sum of squares for each individual model term
print("\n【Individual-Term ANOVA: Type III Partial Sum of Squares】")
anova_terms = sm.stats.anova_lm(ols_model, typ=3)
# Exclude the intercept term
anova_terms = anova_terms.drop("Intercept")
# Rename columns for clarity
anova_terms.rename(columns={"sum_sq": "Partial Sum of Squares (Partial SS)", "df": "Degrees of Freedom df", "F": "F value", "PR(>F)": "p value"}, inplace=True)
display(anova_terms)
print("Note: The results are based on the standard Type III "
    "partial sum of squares, with the intercept excluded. "
    "Each term is evaluated while accounting for the other "
    "terms in the fitted model.")


In [ ]:
# ====================== 8. Residual analysis ======================
print("\n【Residual Analysis】")
print(f"Mean residual      = {np.mean(residual):.6e}")
print(f"Residual standard deviation    = {np.std(residual, ddof=df_error):.6f}")
print(f"Minimum residual    = {np.min(residual):.6f}")
print(f"Maximum residual    = {np.max(residual):.6f}")
w_stat, p_shapiro = stats.shapiro(residual)
print(f"Shapiro-Wilk normality test: W = {w_stat:.4f}, p = {p_shapiro:.4f}")

In [ ]:
# ====================== 9. 3D experimental scatter plot and fitted response surface ======================
import os
# Enable interactive 3D plotting for rotation and zooming

fig1 = plt.figure(figsize=(12, 8))
ax = fig1.add_subplot(111, projection="3d")

# Plot experimental data points
ax.scatter(df["x"], df["y"], df["z"], c="red", s=60, edgecolor="black", label="实验数据点")

# Generate a grid for plotting the fitted response surface
x_grid = np.linspace(df["x"].min(), df["x"].max(), 50)
y_grid = np.linspace(df["y"].min(), df["y"].max(), 50)
Xg, Yg = np.meshgrid(x_grid, y_grid)
# Calculate the fitted surface using the regression coefficients
Zg = coef["Intercept"] + coef["x"]*Xg + coef["y"]*Yg + coef["x2"]*Xg**2 + coef["y2"]*Yg**2 + coef["xy"]*Xg*Yg

surf = ax.plot_surface(Xg, Yg, Zg, alpha=0.6, cmap="viridis", linewidth=0)
fig1.colorbar(surf, shrink=0.5, label="Predicted Cap.")

ax.set_xlabel("PC1 (x)", fontsize=12)
ax.set_ylabel("PC2 (y)", fontsize=12)
ax.set_zlabel("Cap.", fontsize=12)
ax.set_title("Quadratic Response Surface Fitting: "
    "Experimental Data and Fitted Surface", fontsize=14)
ax.legend()
plt. show()
